In [47]:
import pandas as pd

manifest = pd.read_csv("MRI_MANIFEST_CLEANED.csv")
diagnosis = pd.read_csv("All_Subjects_DXSUM_14Aug2025.csv")
diagnosis.rename(columns={"PTID":"subject_id"}, inplace=True)
subject_ids = pd.read_csv("subjects_with_paths.csv", header=None)
subject_ids.columns = ["path", "subject_id", "to_txt"]

In [48]:
merged_df = manifest.merge(diagnosis[['subject_id', 'DIAGNOSIS', 'DXAD']], on='subject_id', how='left')

In [49]:
present_ids = []

for sid in merged_df['subject_id'].unique():
    if subject_ids['subject_id'].str.startswith(sid).any():
        present_ids.append(sid)

len(present_ids)


456

In [50]:
merged_df['subject_id'].nunique(), len(present_ids)

(658, 456)

In [51]:
print(present_ids)

['003_S_1122', '114_S_0416', '073_S_0089', '003_S_0908', '007_S_1222', '041_S_0679', '011_S_0021', '126_S_0605', '141_S_0767', '057_S_0934', '002_S_1280', '127_S_1427', '002_S_0413', '002_S_1155', '022_S_0004', '011_S_0005', '022_S_0007', '011_S_0008', '011_S_0016', '067_S_0019', '011_S_0022', '023_S_0030', '023_S_0031', '035_S_0033', '099_S_0040', '007_S_0041', '023_S_0042', '022_S_0044', '011_S_0023', '035_S_0048', '099_S_0051', '011_S_0053', '067_S_0056', '018_S_0055', '067_S_0059', '023_S_0058', '099_S_0060', '023_S_0061', '022_S_0066', '007_S_0068', '023_S_0078', '067_S_0076', '023_S_0083', '067_S_0077', '023_S_0084', '027_S_0074', '023_S_0093', '099_S_0090', '022_S_0096', '007_S_0101', '020_S_0097', '023_S_0081', '130_S_0102', '099_S_0111', '127_S_0112', '023_S_0126', '022_S_0130', '027_S_0116', '023_S_0139', '131_S_0123', '007_S_0070', '098_S_0149', '007_S_0128', '067_S_0110', '098_S_0160', '027_S_0118', '128_S_0138', '035_S_0156', '128_S_0135', '128_S_0167', '098_S_0171', '011_

In [52]:
diagnosis_df = (
    merged_df[merged_df['subject_id'].isin(present_ids)]
    .groupby('subject_id')['DXAD']
    .unique()
    .reset_index()
)


In [53]:
len(diagnosis_df)

456

In [54]:
# Make a copy so we don't mess up original
subject_ids = subject_ids.copy()
subject_ids['alzheimer_or_not'] = None

# Count occurrences as we go
occurrence_counter = {}

for _, row in merged_df.iterrows():
    sid = row['subject_id']
    dx = row['DXAD']
    
    # How many times we've seen this subject so far
    occurrence_counter[sid] = occurrence_counter.get(sid, 0) + 1
    occ = occurrence_counter[sid]
    
    # Build the matching ID for subject_ids
    if occ == 1:
        target_id = sid
    else:
        target_id = f"{sid}_{occ}"
    
    # If that target exists in subject_ids, assign the diagnosis
    mask = subject_ids['subject_id'] == target_id
    if mask.any():
        subject_ids.loc[mask, 'alzheimer_or_not'] = dx


In [55]:
subject_ids['alzheimer_or_not'].value_counts()

alzheimer_or_not
-4.0    616
 1.0    140
Name: count, dtype: int64

In [61]:
subject_ids.loc[subject_ids['alzheimer_or_not'] == -4, 'alzheimer_or_not'] = 0


In [62]:
subject_ids['alzheimer_or_not'].value_counts()

alzheimer_or_not
0      616
1.0    140
Name: count, dtype: int64

In [63]:
subject_ids.to_csv("subjects_with_alzheimers_labels.csv", index=False)

In [64]:
subject_ids[subject_ids['subject_id'] != 0].value_counts()

path                                                                                                                        subject_id    to_txt                                                                                                                                                                                     alzheimer_or_not
MRIs_IMAGES_FILTERED_NOMASKS/ADNI_941_S_1311_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20081026142433381_S56646_I123816.nii  941_S_1311_2  941_S_1311_2=/home/e.yavarova/alzheimers_research_lala_emiliya/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_941_S_1311_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20081026142433381_S56646_I123816.nii  0.0                 1
MRIs_IMAGES_FILTERED_NOMASKS/ADNI_002_S_0413_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20070319120107557_S13894_I45121.nii   002_S_0413    002_S_0413=/home/e.yavarova/alzheimers_research_lala_emiliya/MRIs_IMAGES_FILTERED_NOMASKS/ADNI_002_S_0413_MR_MPR-R__GradWarp__B1_Correction__N3_Br_20070319120107557_S13894_I

In [81]:
# Filter merged_df to only present IDs
filtered_df = merged_df[merged_df['subject_id'].isin(present_ids)]

# Identify rows (assuming PTGENDER: 1 = male, 2 = female)
male_rows_ad = filtered_df[(filtered_df['PTGENDER'] == 1) & (filtered_df['DXAD'] == 1)]
female_rows_ad = filtered_df[(filtered_df['PTGENDER'] == 2) & (filtered_df['DXAD'] == 1)]
male_rows_no_ad = filtered_df[(filtered_df['PTGENDER'] == 1) & (filtered_df['DXAD'] != 1)]
female_rows_no_ad = filtered_df[(filtered_df['PTGENDER'] == 2) & (filtered_df['DXAD'] != 1)]

# Randomly select rows
yes_AD_man = male_rows_ad.sample(n=5, random_state=42).index
no_AD_man = male_rows_no_ad.sample(n=5, random_state=42).index
yes_AD_woman = female_rows_ad.sample(n=5, random_state=42).index
no_AD_woman = female_rows_no_ad.sample(n=5, random_state=42).index

# Combine all sampled rows into one DataFrame
sampled_df = pd.concat([
    male_rows_ad.loc[yes_AD_man],
    male_rows_no_ad.loc[no_AD_man],
    female_rows_ad.loc[yes_AD_woman],
    female_rows_no_ad.loc[no_AD_woman]
])

# Reset the index
sampled_df = sampled_df.reset_index(drop=True)


In [82]:
male_rows_ad.loc[yes_AD_man]["subject_id"].unique()


array(['023_S_0388', '022_S_1394', '016_S_1326', '114_S_0979',
       '036_S_1001'], dtype=object)

In [83]:
female_rows_ad.loc[yes_AD_woman]["subject_id"].unique()

array(['023_S_1247', '067_S_1253', '036_S_0869', '051_S_1331',
       '057_S_0474'], dtype=object)

In [84]:
male_rows_no_ad.loc[no_AD_man]["subject_id"].unique()

array(['052_S_1352', '032_S_0677', '022_S_0096', '128_S_1408'],
      dtype=object)

In [85]:
female_rows_no_ad.loc[no_AD_woman]["subject_id"].unique()

array(['141_S_0767', '053_S_0389', '003_S_1122', '131_S_0441',
       '023_S_0331'], dtype=object)

In [86]:
import os
import shutil
import pandas as pd

# Example: your dataframe
# df = pd.read_csv("subject_ids_with_dx.csv")  # must have a column "subject_id"
subject_ids = sampled_df["subject_id"].tolist()

# Paths
source_root = "/home/e.yavarova/alzheimers_research_lala_emiliya/FASTSURFER_MASKS"   # folder where all subject subfolders live
target_root = "/home/e.yavarova/alzheimers_research_lala_emiliya/selected_subjects"  # where to copy

os.makedirs(target_root, exist_ok=True)

for sid in subject_ids:
    src_subject = os.path.join(source_root, sid)        # e.g. /source/127_S_0844
    src_mri = os.path.join(src_subject, "mri")          # e.g. /source/127_S_0844/mri
    dst_mri = os.path.join(target_root, sid, "mri")     # e.g. /selected_subjects/127_S_0844/mri

    if os.path.exists(src_mri):
        print(f"Copying MRI files for {sid} -> {dst_mri}")
        os.makedirs(os.path.dirname(dst_mri), exist_ok=True)
        shutil.copytree(src_mri, dst_mri, dirs_exist_ok=True)
    else:
        print(f"⚠️ MRI folder not found for {sid}: {src_mri}")


Copying MRI files for 023_S_0388 -> /home/e.yavarova/alzheimers_research_lala_emiliya/selected_subjects/023_S_0388/mri
Copying MRI files for 022_S_1394 -> /home/e.yavarova/alzheimers_research_lala_emiliya/selected_subjects/022_S_1394/mri
Copying MRI files for 016_S_1326 -> /home/e.yavarova/alzheimers_research_lala_emiliya/selected_subjects/016_S_1326/mri
Copying MRI files for 114_S_0979 -> /home/e.yavarova/alzheimers_research_lala_emiliya/selected_subjects/114_S_0979/mri
Copying MRI files for 036_S_1001 -> /home/e.yavarova/alzheimers_research_lala_emiliya/selected_subjects/036_S_1001/mri
Copying MRI files for 052_S_1352 -> /home/e.yavarova/alzheimers_research_lala_emiliya/selected_subjects/052_S_1352/mri
Copying MRI files for 032_S_0677 -> /home/e.yavarova/alzheimers_research_lala_emiliya/selected_subjects/032_S_0677/mri
Copying MRI files for 022_S_0096 -> /home/e.yavarova/alzheimers_research_lala_emiliya/selected_subjects/022_S_0096/mri
Copying MRI files for 128_S_1408 -> /home/e.yava

In [87]:
sampled_df.head(20)

,Unnamed: 0,subject_id,visit,PTGENDER,ARM,IHSYMPTOM,MHPSYCH,MH2NEURL,CENROLL,DIAGNOSIS,DXAD
0,9824,023_S_0388,sc,1.0,8.0,NaN,1.0,0.0,1.0,3.0,1.0
1,15762,022_S_1394,m48,1.0,5.0,NaN,1.0,0.0,1.0,3.0,1.0
2,15409,016_S_1326,m06,1.0,8.0,NaN,NaN,NaN,NaN,3.0,1.0
3,13244,114_S_0979,m12,1.0,6.0,NaN,NaN,NaN,NaN,3.0,1.0
4,13879,036_S_1001,m18,1.0,6.0,NaN,NaN,NaN,NaN,3.0,1.0
5,15613,052_S_1352,m96,1.0,2.0,NaN,NaN,NaN,NaN,1.0,NaN
6,11487,032_S_0677,m12,1.0,7.0,NaN,NaN,NaN,NaN,1.0,NaN
7,8024,022_S_0096,4_init,1.0,4.0,NaN,NaN,NaN,NaN,1.0,NaN
8,15829,128_S_1408,m30,1.0,5.0,NaN,NaN,NaN,NaN,2.0,-4.0
9,8041,022_S_0096,m48,1.0,4.0,NaN,NaN,NaN,NaN,1.0,NaN


In [89]:
sampled_df.drop(columns=["Unnamed: 0", "visit", "ARM", "IHSYMPTOM", "MHPSYCH", "MH2NEURL", "CENROLL", "DIAGNOSIS"], inplace=True)

In [96]:
sampled_df.loc[sampled_df['DXAD'] != 1, 'DXAD'] = 0
sampled_df.loc[sampled_df['PTGENDER'] != 1, 'PTGENDER'] = "female"
sampled_df.loc[sampled_df['PTGENDER'] == 1, 'PTGENDER'] = "male"

/tmp/ipykernel_3101707/3658046731.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'female' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  sampled_df.loc[sampled_df['PTGENDER'] != 1, 'PTGENDER'] = "female"


In [97]:
sampled_df.head(20)

,subject_id,PTGENDER,DXAD
0,023_S_0388,male,1.0
1,022_S_1394,male,1.0
2,016_S_1326,male,1.0
3,114_S_0979,male,1.0
4,036_S_1001,male,1.0
5,052_S_1352,male,0.0
6,032_S_0677,male,0.0
7,022_S_0096,male,0.0
8,128_S_1408,male,0.0
9,022_S_0096,male,0.0


In [100]:
sampled_df.to_csv("20_selected_mris_fastsurfer.csv", index=False)


NotWritableError: The current user does not have write permissions to a required path.
  path: /opt/miniconda3/pkgs/cache/9e0f62c3.json
  uid: 808209227
  gid: 808200513

If you feel that permissions on this path are set incorrectly, you can manually
change them by executing

  $ sudo chown 808209227:808200513 /opt/miniconda3/pkgs/cache/9e0f62c3.json

In general, it's not advisable to use 'sudo conda'.


